# Proyek Klasifikasi Gambar CIFAR-10

Notebook ini menggunakan dataset CIFAR-10 yang berisi 60.000 gambar dari 10 kelas. Dataset ini bukan Rock Paper Scissors dan bukan X-Ray, sehingga sesuai dengan batasan submission. Data dibagi menjadi train, validation, dan test set, lalu model diekspor ke SavedModel, TF-Lite, dan TFJS.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
BATCH_SIZE = 64
EPOCHS = 40
IMAGE_SIZE = (32, 32)
OUTPUT_DIR = Path('.')

CLASS_NAMES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

## Load Dataset dan Split Data

CIFAR-10 memiliki 50.000 gambar training dan 10.000 gambar testing. Dari data training, 20% dipisahkan menjadi validation set.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full.reshape(-1),
)

print('Total gambar:', len(x_train_full) + len(x_test))
print('Train:', len(x_train))
print('Validation:', len(x_val))
print('Test:', len(x_test))
print('Jumlah kelas:', len(CLASS_NAMES))
assert len(x_train_full) + len(x_test) >= 1000

In [ ]:
def make_dataset(images, labels, shuffle=False):
    labels = labels.reshape(-1).astype('int64')
    ds = tf.data.Dataset.from_tensor_slices((images.astype('float32') / 255.0, labels))
    if shuffle:
        ds = ds.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(x_train, y_train, shuffle=True)
val_ds = make_dataset(x_val, y_val)
test_ds = make_dataset(x_test, y_test)

## Model Sequential CNN

Model memakai `Sequential`, beberapa layer `Conv2D`, dan pooling layer sesuai kriteria submission.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.40),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.3),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

## Evaluasi Model

In [ ]:
train_loss, train_acc = model.evaluate(train_ds)
test_loss, test_acc = model.evaluate(test_ds)
print(f'Train accuracy: {train_acc:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

## Plot Akurasi dan Loss

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig('accuracy_loss_plot.png', dpi=160)
plt.show()

## Export SavedModel, TF-Lite, dan TFJS

In [ ]:
tf.saved_model.save(model, 'saved_model')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
Path('tflite').mkdir(exist_ok=True)
Path('tflite/model.tflite').write_bytes(tflite_model)

!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model saved_model tfjs_model

## Inference Menggunakan TF-Lite

In [ ]:
interpreter = tf.lite.Interpreter(model_path='tflite/model.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample_index = 0
sample_image = x_test[sample_index].astype('float32') / 255.0
sample_batch = np.expand_dims(sample_image, axis=0).astype(input_details[0]['dtype'])

interpreter.set_tensor(input_details[0]['index'], sample_batch)
interpreter.invoke()
prediction = interpreter.get_tensor(output_details[0]['index'])[0]
predicted_class = CLASS_NAMES[int(np.argmax(prediction))]
actual_class = CLASS_NAMES[int(y_test[sample_index][0])]

plt.imshow(x_test[sample_index])
plt.axis('off')
plt.title(f'Prediksi: {predicted_class} | Aktual: {actual_class}')
plt.show()
print('Prediksi:', predicted_class)
print('Aktual:', actual_class)
print('Confidence:', float(np.max(prediction)))

In [ ]:
metadata = {
    'dataset': 'CIFAR-10',
    'total_images': int(len(x_train_full) + len(x_test)),
    'classes': CLASS_NAMES,
    'train_images': int(len(x_train)),
    'validation_images': int(len(x_val)),
    'test_images': int(len(x_test)),
    'train_accuracy': float(train_acc),
    'test_accuracy': float(test_acc),
    'tflite_inference_example': {
        'predicted_class': predicted_class,
        'actual_class': actual_class,
        'confidence': float(np.max(prediction)),
    },
}
Path('metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
metadata